In [1]:
import warnings

# Ignore FutureWarnings and warnings related to inplace operations
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*inplace.*")

import yfinance as yf
import pandas as pd

Collects daily OHLCV price data and bid-ask spread proxy for ticker for that year range across a specified year range. Simplifies the process of data gathering, given that instead of collecting an X number of companies per year it returns every company in the form of a DataFrame. Spread_Proxy column is used to verify that Hendershott, Jones & Menkveld (2011) experiment that AT adoption causes bid-ask spread to narrow.
    
**Algorithm:**
1. Downloads all firms' stock daily movement data OHLCV from a start of the year to the end.
2. Concatenates column for daily returns of firm tickers.
3. Calculate and then concatenates spread proxy into the DataFrame, using the scaled high-low range formula.
4. What occurs to the Ticker symbols is that they become Row Levels with the column.
5. Pivots the Ticker Row Level into Column Level.

**Parameters:**
- ticker (yf.Ticker): Stock ticker symbol already converted to yf format e.g. 'AAPL'
- year (int) : Desired year for data collection
- first_qe (pd.Timestamp): when first quarterly earnings date
- interval (str): interval of analysis for stock ticker e.g. hourly or daily

**Output:**: DataFrame with column values of open, high, low, volume, daily return, spread proxy (from a row range from first quarterly earnings date to end of year)

In [2]:
def collect_stock_data(tickers: list[str],
                        year: int,
                        interval: str = "1d"
                        ) -> pd.DataFrame | None:
    """
    Downloads price data for multiple tickers over a given year,
    calculates daily returns and a bid-ask spread proxy, and
    reshapes the result into long format (Date, Ticker as rows).

    Returns:
        pd.DataFrame in long format with columns:
        Open, High, Low, Close, Volume, Return, Spread_Proxy
        indexed by (Date, Ticker)
    """
    try:
        start_date = pd.Timestamp(f'{year}-01-01')
        end_date = pd.Timestamp(f'{year+1}-01-1')

        df = yf.download(tickers, start=start_date, end=end_date,
                          interval=interval, auto_adjust=True,
                          progress=False)
        # SAFETY CHECK: Checks whether Yahoo Finance has the data avaliable for tickers
        if df.empty:
            print(f"No data returned for {tickers} in {year}")
            return None

        # Daily returns per ticker
        returns = df['Close'].pct_change()
        returns.columns = pd.MultiIndex.from_product(
            [['Return'], returns.columns], names=['Field', 'Ticker']
        )
        df = pd.concat([df, returns], axis=1)

        # Bid-ask spread proxy per ticker
        spread_proxy = (df['High'] - df['Low']) / df['Close']
        spread_proxy.columns = pd.MultiIndex.from_product(
            [['Spread_Proxy'], spread_proxy.columns], names=['Field', 'Ticker']
        )
        df = pd.concat([df, spread_proxy], axis=1)

        # Reshape from wide MultiIndex to long format
        df_long = df.stack(level='Ticker', future_stack=True)

        return df_long

    except Exception as e:
        print(f"Failed to retrieve data for {tickers}: {e}")
        return None

Testing **collect_stock_data()** function

In [3]:
ticker = ['AAPL', 'MSFT','META']
target_year = 2024

stock_data: pd.DataFrame = collect_stock_data(ticker, target_year)
print(stock_data)

                        Close        High         Low        Open    Volume  \
Date       Ticker                                                             
2024-01-02 AAPL    183.404007  186.170285  181.675085  184.895814  82488700   
           META    343.004822  349.809643  336.784400  347.987102  19042200   
           MSFT    363.117889  368.042749  359.103584  366.045381  25258600   
2024-01-03 AAPL    182.030746  183.641118  181.220616  182.001109  58414500   
           META    341.202118  344.649115  339.924347  341.707289  15451100   
...                       ...         ...         ...         ...       ...   
2024-12-30 META    587.868042  593.535546  582.240349  585.392253   7025900   
           MSFT    419.060455  421.743517  416.170254  420.273762  13158700   
2024-12-31 AAPL    248.615799  251.455194  247.632926  250.621250  39480700   
           META    582.170837  590.582551  580.520270  588.892295   6019500   
           MSFT    415.775696  420.934679  414.94710

c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/w

Gathers Earnings dates for multiples tickers (formatted in an array of strings) for a year (e.g. 2008). Helps simplify the process of earnings date gather, where the data of all tickers' within the year are gather all at once and placed into a DataFrame instead of being gathered one-by-one.

**Algorithm:**
1. Loops through each ticker within the **tickers** array to get its earnings dates within the specified **year** parameter.
2. Adds "Ticker" and "Quarter" tags as the Rows Levels of the Table.
3. Removes the EPS columns.
4. Changes DataFrame format from wide table to long.

**Parameters:**
1. tickers: array of stock tickers e.g. ['MSFT', 'AAPL', 'META']
2. year: desired year range for earnings data gathering

**Output:** stack DataFrame formatted long with row levels as Ticker Quarter and Column as Date

In [4]:
def tickers_earnings_long(
    tickers: list[str],
    year: int
) -> pd.DataFrame | None:
    """
    Retrieves earnings dates for multiple tickers and returns
    them in long format: one row per ticker-quarter event.
    
    Long format columns:
        earnings_date (index), ticker, quarter, timing
    """
    earnings_data = []

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            earnings_df = stock.get_earnings_dates(limit=100)

            # SAFETY CHECK: Checks whether earnings_df is empty
            if earnings_df is None or earnings_df.empty:
                print(f"* {ticker}: no earnings data available")
                continue

            qe_df = earnings_df[earnings_df.index.year == year].copy()

            # SAFETY CHECK: Checks whether qe_df is empty
            if qe_df.empty:
                print(f"* {ticker}: no earnings in {year}")
                continue

            # Tag with Ticker and Quarter — long format identifiers
            qe_df['Ticker'] = ticker
            qe_df['Quarter'] = qe_df.index.quarter.map(
                {1: '1', 2: '2', 3: '3', 4: '4'}
            )

            # Delete EPS columns
            del qe_df['EPS Estimate'], qe_df['Reported EPS'], qe_df['Surprise(%)']
            
            earnings_data.append(qe_df)

        except Exception as e:
            print(f"* {ticker}: failed — {e}")
            continue

    if not earnings_data:
        return None

    # Stack every ticker's rows into one long table
    long_table = pd.concat(earnings_data, ignore_index=False)

    
    return long_table.sort_values(by=['Ticker', 'Quarter'])

Testing **get_multiple_tickers_earnings_long()** function

In [5]:
tickers = ['MSFT', 'NVDA', 'META']
qe_table = tickers_earnings_long(tickers, 2019)
print(qe_table)

c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\base.py:716: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Surprise (%)': 'Surprise(%)'}, inplace=True)
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\base.py:716: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Surprise (%)': 'Surprise(%)'}, inplace=True)


                          Ticker Quarter
Earnings Date                           
2019-01-30 16:00:00-05:00   META       1
2019-04-24 16:00:00-04:00   META       2
2019-07-24 16:00:00-04:00   META       3
2019-10-30 16:00:00-04:00   META       4
2019-01-30 16:00:00-05:00   MSFT       1
2019-04-24 16:00:00-04:00   MSFT       2
2019-07-18 16:00:00-04:00   MSFT       3
2019-10-23 16:00:00-04:00   MSFT       4
2019-02-14 16:00:00-05:00   NVDA       1
2019-05-16 16:00:00-04:00   NVDA       2
2019-08-15 16:00:00-04:00   NVDA       3
2019-11-14 16:00:00-05:00   NVDA       4


c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\base.py:716: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Surprise (%)': 'Surprise(%)'}, inplace=True)


Getting historical standard deviations of multiple tickers 365 days before the 4 quarterly earnings announcement. It does so by simply extracting the a long table from get_multiple_tickers_earnings_long() and adding an additional column of Historical Standard Deviation.

**Algorithm:**
1. Loops through each ticker to get their earnings data
2. Uses ticker's quarters' to calculate for the historical s.d. 365 days before earnings date
3. Appends data into results array
4. Finally, initializes n tickers stored in *results* to a DataFrame

**Parameters:**
1. tickers: list of ticker strings
2. earnings_dates: DataFrame

**Output:** pd.DataFrame with column of Earnings Date, Ticker, Quarter, and Historical SD


In [6]:
def tickers_historical_sd(
        tickers: list[str],
        earnings_dates: pd.DataFrame,
        interval: str = '1d',
    ) -> pd.DataFrame:
    """
    Calculates the historical standard deviation (s.d.) of daily
    returns for multiple tickers, using a 365-calendar-day window
    ending the day before each respective quarterly earnings call.

    Returns:
        pd.DataFrame with columns: Ticker, Quarter, Earnings Date,
        Historical Standard Deviation
    """
    results = []

    for ticker in tickers:
        # Gets earnings dates for each ticker
        ticker_qe_dates = earnings_dates[earnings_dates['Ticker'] == ticker]
        
        for quarter in range(1, 5):
            try:
                # Looking to specific Quarter
                quarter_match = ticker_qe_dates[ticker_qe_dates['Quarter'] == str(quarter)].index

                if len(quarter_match) == 0:
                    print(f"* No earnings date for {ticker} Q{quarter}")
                    continue

                date = quarter_match[0]

                start_date = date - pd.Timedelta(days=365)
                end_date = date

                df = yf.download(
                    ticker, start=start_date, end=end_date,
                    interval=interval, auto_adjust=True, progress=False
                )

                if df.empty:
                    print(f"* No price data for {ticker} Q{quarter}")
                    continue

                sd_val = df['Close'].pct_change().std()

                results.append({
                    'Earnings Date': date,
                    'Ticker': ticker,
                    'Quarter': quarter,
                    'Historical SD': sd_val   # Historical Standard Deviation (SD)
                })

            except Exception as e:
                print(f"* {ticker} Q{quarter}: {e}")
                continue
    historical_sd = pd.DataFrame(results)
    return historical_sd

Testing **multiple_ticker_historical_sd()** Function

In [7]:
tickers = ['MSFT', 'NVDA', 'META']
year = 2020

earnings_dates = tickers_earnings_long(tickers, year)
data = tickers_historical_sd(tickers, earnings_dates)
print(data)

c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\base.py:716: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Surprise (%)': 'Surprise(%)'}, inplace=True)
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\base.py:716: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'Surprise (%)': 'Surprise(%)'}, inplace=True)
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\base.py:716: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8)

               Earnings Date Ticker  Quarter  \
0  2020-01-29 16:00:00-05:00   MSFT        1   
1  2020-04-29 16:00:00-04:00   MSFT        2   
2  2020-07-22 16:00:00-04:00   MSFT        3   
3  2020-10-27 16:00:00-04:00   MSFT        4   
4  2020-02-13 16:00:00-05:00   NVDA        1   
5  2020-05-21 16:00:00-04:00   NVDA        2   
6  2020-08-19 16:00:00-04:00   NVDA        3   
7  2020-11-18 16:00:00-05:00   NVDA        4   
8  2020-01-29 16:00:00-05:00   META        1   
9  2020-04-29 17:00:00-04:00   META        2   
10 2020-07-30 16:00:00-04:00   META        3   
11 2020-10-29 16:00:00-04:00   META        4   

                             Historical SD  
0    Ticker
MSFT    0.01185
dtype: float64  
1   Ticker
MSFT    0.025204
dtype: float64  
2   Ticker
MSFT    0.025718
dtype: float64  
3   Ticker
MSFT    0.026948
dtype: float64  
4   Ticker
NVDA    0.022354
dtype: float64  
5   Ticker
NVDA    0.035029
dtype: float64  
6    Ticker
NVDA    0.03449
dtype: float64  
7   Ticker
NVDA

c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
c:\Users\John Culpepper\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo